# S30 EA2 Reconocimiento de un dataset de ventas de cafetería

**Estudiante:** José David Osorio Gallego  
**Asignatura:** Analítica de datos | Proyecto integrador  
**Propósito:** reconocer los registros de transacciones, su estructura y sus posibles usos en una pregunta descriptiva.

**Origen:** [adityasuyal/cafe-sales-cleaned](https://huggingface.co/datasets/adityasuyal/cafe-sales-cleaned). La ficha describe datos **sintéticos**. El negocio del proyecto es un escenario hipotético, no una empresa estudiada.

La versión empleada es el archivo CSV limpio completo de la revisión indicada más abajo (9.540 registros de 2023 según la ficha). Esta fase **no** busca demostrar conclusiones comerciales, sino comprobar qué información hay disponible.


## 1 Obtener los datos de Hugging Face

El CSV es público y se descarga mediante código desde la revisión fijada del repositorio. Se utiliza la versión limpia completa; no se hace muestreo. En el visor del dataset aparece como configuración única, split `train`.


In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import Request, urlopen
import pandas as pd

DATASET_ID = "adityasuyal/cafe-sales-cleaned"
REVISION = "f4e6dfa1fe791ba84807921c949bd84f02b4bc69"
ARCHIVO = "cleaned_cafe_sales.csv"
SPLIT = "train (archivo CSV completo)"
URL = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{REVISION}/{ARCHIVO}"

# Descarga reproducible desde Hugging Face; no utiliza una copia privada ni requiere token.
peticion = Request(URL, headers={"User-Agent": "S30-EA2-academic-notebook/1.0"})
with urlopen(peticion, timeout=90) as respuesta:
    contenido_csv = respuesta.read()
df = pd.read_csv(BytesIO(contenido_csv))

print(f"Dataset: {DATASET_ID}")
print(f"Revisión: {REVISION}")
print(f"Archivo/split: {ARCHIVO} / {SPLIT}")
print(f"DataFrame cargado: {len(df):,} registros")


Dataset: adityasuyal/cafe-sales-cleaned
Revisión: f4e6dfa1fe791ba84807921c949bd84f02b4bc69
Archivo/split: cleaned_cafe_sales.csv / train (archivo CSV completo)
DataFrame cargado: 9,540 registros


## 2 Exploración inicial del DataFrame

Cada fila es un registro de transacción con un producto, cantidad, precio unitario, total, medio de pago, modalidad de atención y fecha. La ficha define el importe monetario en USD. Se observan los primeros registros, la estructura completa y los nulos sin modificar el DataFrame de origen.


In [2]:
print("Primeros 5 registros:")
print(df.head(5).to_string(index=False))
print("\nDimensiones (filas, columnas):", df.shape)
print("\nNombres de las columnas:")
print(list(df.columns))


Primeros 5 registros:
Transaction ID   Item  Quantity  Price Per Unit  Total Spent Payment Method Location Transaction Date
   TXN_1961373 Coffee       2.0             2.0          4.0    Credit Card Takeaway       2023-09-08
   TXN_4977031   Cake       4.0             3.0         12.0           Cash In-store       2023-05-16
   TXN_4271903 Cookie       4.0             1.0          4.0    Credit Card In-store       2023-07-19
   TXN_7034554  Salad       2.0             5.0         10.0 Digital Wallet  Unknown       2023-04-27
   TXN_3160411 Coffee       2.0             2.0          4.0 Digital Wallet In-store       2023-06-11

Dimensiones (filas, columnas): (9540, 8)

Nombres de las columnas:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [3]:
print("Tipos de datos por columna:")
print(df.dtypes.to_string())
print("\nValores nulos por columna:")
print(df.isna().sum().to_string())


Tipos de datos por columna:
Transaction ID       object
Item                 object
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object

Valores nulos por columna:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0


## 3 Lectura responsable de los datos

La necesidad propuesta es que una cafetería hipotética conozca **cómo se distribuyen las unidades vendidas y el importe de ventas por tipo de producto durante 2023**. Las columnas `Item`, `Quantity`, `Total Spent` y `Transaction Date` permiten diseñar esa descripción.

**Precaución:** la categoría literal `Unknown` en `Location` no equivale a un nulo de pandas. Por tanto, una cifra de cero nulos no significa que todos los datos tengan una ubicación conocida. Sin costos no es posible estudiar rentabilidad; al tratarse de registros sintéticos, los resultados no deben generalizarse a negocios reales.


In [4]:
desconocidos = int(df["Location"].eq("Unknown").sum())
print(f"Registros con Location = 'Unknown': {desconocidos:,} ({desconocidos/len(df):.2%})")
print("Periodo registrado:", df["Transaction Date"].min(), "a", df["Transaction Date"].max())
print("Tipos de producto diferentes:", df["Item"].nunique())


Registros con Location = 'Unknown': 3,779 (39.61%)
Periodo registrado: 2023-01-01 a 2023-12-31
Tipos de producto diferentes: 8


## 4 Generación automática del resumen TXT

El resumen se construye **a partir del mismo `df`** explorado arriba. Al volver a ejecutar el notebook se crea la carpeta si hace falta y se reemplaza el archivo con el contenido actualizado. Es válido ejecutarlo desde la raíz del repositorio o desde `AE2/notebooks`.


In [5]:
carpeta_actual = Path.cwd().resolve()
if (carpeta_actual / "AE2").is_dir():
    raiz_repo = carpeta_actual
elif carpeta_actual.name == "notebooks" and carpeta_actual.parent.name == "AE2":
    raiz_repo = carpeta_actual.parent.parent
else:
    raise RuntimeError("Abre Jupyter desde la raíz del repositorio o desde AE2/notebooks.")

salida = raiz_repo / "AE2" / "resultados" / "resumen_dataframe.txt"
salida.parent.mkdir(parents=True, exist_ok=True)

lineas = [
    "RESUMEN DEL DATAFRAME - S30 EA2",
    "Estudiante: José David Osorio Gallego",
    f"Identificador del dataset: {DATASET_ID}",
    f"Archivo: {ARCHIVO}",
    f"Split/configuración: {SPLIT}",
    f"Revisión: {REVISION}",
    "Origen: Hugging Face; versión limpia de datos sintéticos de cafetería",
    "Muestra: no; archivo CSV limpio completo",
    f"Filas: {df.shape[0]}",
    f"Columnas: {df.shape[1]}",
    "",
    "COLUMNAS, TIPOS DE PANDAS Y VALORES NULOS",
]
for nombre in df.columns:
    lineas.append(f"- {nombre} | tipo={df[nombre].dtype} | nulos={int(df[nombre].isna().sum())}")
lineas += [
    "",
    "NOTA DE INTERPRETACIÓN",
    "Los nulos de Pandas no incluyen categorías de texto como 'Unknown'.",
    f"Location = 'Unknown': {int(df['Location'].eq('Unknown').sum())} registros.",
    "Las filas son transacciones sintéticas; no representan un negocio real.",
]
salida.write_text("\n".join(lineas) + "\n", encoding="utf-8")

contenido_verificado = salida.read_text(encoding="utf-8")
assert f"Filas: {df.shape[0]}" in contenido_verificado
assert f"Columnas: {df.shape[1]}" in contenido_verificado
for nombre in df.columns:
    assert f"- {nombre} | tipo={df[nombre].dtype} | nulos={int(df[nombre].isna().sum())}" in contenido_verificado
print(f"Resumen generado y verificado: {salida.relative_to(raiz_repo)}")
print("\n" + contenido_verificado)


Resumen generado y verificado: AE2/resultados/resumen_dataframe.txt

RESUMEN DEL DATAFRAME - S30 EA2
Estudiante: José David Osorio Gallego
Identificador del dataset: adityasuyal/cafe-sales-cleaned
Archivo: cleaned_cafe_sales.csv
Split/configuración: train (archivo CSV completo)
Revisión: f4e6dfa1fe791ba84807921c949bd84f02b4bc69
Origen: Hugging Face; versión limpia de datos sintéticos de cafetería
Muestra: no; archivo CSV limpio completo
Filas: 9540
Columnas: 8

COLUMNAS, TIPOS DE PANDAS Y VALORES NULOS
- Transaction ID | tipo=object | nulos=0
- Item | tipo=object | nulos=0
- Quantity | tipo=float64 | nulos=0
- Price Per Unit | tipo=float64 | nulos=0
- Total Spent | tipo=float64 | nulos=0
- Payment Method | tipo=object | nulos=0
- Location | tipo=object | nulos=0
- Transaction Date | tipo=object | nulos=0

NOTA DE INTERPRETACIÓN
Los nulos de Pandas no incluyen categorías de texto como 'Unknown'.
Location = 'Unknown': 3779 registros.
Las filas son transacciones sintéticas; no representan